# Bước 06/06 — Đánh giá vs baseline V6.2E YOLO11n@640

Baseline: mAP50 0.768 / mAP50:95 0.413 / gap 0.355 (foreign .537/.295, strand .749/.356).

Gate: mAP50:95 toàn cục giảm ≤ 1 điểm; foreign_object AP50 +2 và AP50:95 +1; broken_strand AP50:95 +1. **Pass chỉ mở quyền review/promotion bằng tay** → freeze V6.5 canonical → YOLO11m@1024 → RT-DETR B0.

| | Camber Stash |
|---|---|
| **Input** | `stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation/v6p5_y11n_ablation_<12hex>/v6p5_yolo11n_data_ablation_report.json` |
| **Output** | `stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_data_ablation/v6p5_y11n_ablation_<12hex>/v6p5-abl01-yolo11n-640-e50_vs_baseline.csv` |

Chạy lại notebook bất cứ lúc nào: bước đã xong + đúng config sẽ được **reuse**, bước dở dang sẽ **resume**.
Bước tiếp theo: **quyết định promotion (người làm)**

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
# ===== DATASET TRÊN CAMBER — điền tag nếu muốn chọn tay, None = tự động =====
PARENT_DATASET = "v6p2e_23562a45b343"   # V6.2E: cố định, KHÔNG đổi
EXTERNAL_STAGE = None   # vd "v6p5_ext_0123456789ab"     (tạo ở bước 03)
COMPOSE        = None   # vd "v6p5_compose_0123456789ab" (tạo ở bước 04)
REQUIRE_CAMBER = True   # lưu chính trên Camber -> dừng nếu chưa đăng nhập

from yolov11sdi.camber import CamberClient
from yolov11sdi.environment import load_camber_api_key
from yolov11sdi.reporting import status
cfg = pipeline.load(stage_tag=EXTERNAL_STAGE, compose_tag=COMPOSE, parent_tag=PARENT_DATASET)
load_camber_api_key(paths.env)
cam = CamberClient(paths, cfg.get("camber.stash_prefix"), cfg.runtime("camber_bin"))
if REQUIRE_CAMBER and not cam.available():
    raise RuntimeError("Camber chưa sẵn sàng: chạy `camber login` hoặc export CAMBER_API_KEY "
                       "(CAMBER_BIN=... nếu CLI không nằm trong PATH). Chỉ chạy local: REQUIRE_CAMBER=False")
display(status.dataset_card(cfg, paths, cam))

In [ ]:
outcome = pipeline.run_stage("evaluate", cfg, force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
cmp_csv = paths.exports / f"{cfg.get('training.experiment')}_vs_baseline.csv"
if cmp_csv.exists():
    display(pd.read_csv(cmp_csv))

Nếu foreign_object tăng nhưng broken_strand giảm → tách ablation theo nguồn (`abl02` chỉ foreign, `abl03` chỉ strand) trước khi freeze bất cứ thứ gì.